# Data Preprocessing, Validation, and Frozen Backbone Construction

## 1. Environment Setup and Raw Data Download

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
%cd /content/drive/MyDrive/football_lifecycle

!curl -L "https://raw.githubusercontent.com/salimt/football-datasets/main/datalake/transfermarkt/player_profiles/player_profiles.csv" -o data/raw/player_profiles.csv
!curl -L "https://raw.githubusercontent.com/salimt/football-datasets/main/datalake/transfermarkt/player_market_value/player_market_value.csv" -o data/raw/player_market_value.csv

/content/drive/MyDrive/football_lifecycle
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 25.1M  100 25.1M    0     0  21.9M      0  0:00:01  0:00:01 --:--:-- 21.9M
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 22.7M  100 22.7M    0     0  18.3M      0  0:00:01  0:00:01 --:--:-- 18.3M


## 2. Preprocessing Utility Functions

In [3]:
%%writefile /content/drive/MyDrive/football_lifecycle/src/a_preprocessing_utils.py
from pathlib import Path
import re
import numpy as np
import pandas as pd


def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace("-", "_", regex=False)
        .str.replace(r"[^0-9a-zA-Z_]", "", regex=True)
    )
    return df


def pick_first_existing(df: pd.DataFrame, candidates, required=True):
    for col in candidates:
        if col in df.columns:
            return col
    if required:
        raise KeyError(f"None of these columns were found: {candidates}")
    return None


def parse_date_flex(series: pd.Series) -> pd.Series:
    return pd.to_datetime(series, errors="coerce")


def parse_unix_flex(series: pd.Series) -> pd.Series:
    s = pd.to_numeric(series, errors="coerce")
    non_null = s.dropna()
    if non_null.empty:
        return pd.to_datetime(series, errors="coerce")
    median_val = non_null.median()
    unit = "ms" if median_val > 1e12 else "s"
    return pd.to_datetime(s, unit=unit, errors="coerce")


def to_nullable_int(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce").astype("Int32")


def extract_season_start_year(series: pd.Series) -> pd.Series:
    s = series.astype(str).str.strip()

    # First try 4-digit year, such as 2021 or 2021/22
    four_digit = s.str.extract(r"(\d{4})", expand=False)
    out = pd.to_numeric(four_digit, errors="coerce")

    # If still missing, try 2-digit season like 21/22 -> 2021
    mask = out.isna()
    two_digit = s[mask].str.extract(r"(\d{2})/\d{2}", expand=False)
    two_digit_num = pd.to_numeric(two_digit, errors="coerce")
    out.loc[mask] = np.where(two_digit_num.notna(), 2000 + two_digit_num, np.nan)

    return out.astype("Int32")


def map_broad_position(raw_position: pd.Series, sub_position: pd.Series = None) -> pd.Series:
    raw = raw_position.fillna("").astype(str).str.lower().str.strip()

    if sub_position is not None:
        sub = sub_position.fillna("").astype(str).str.lower().str.strip()
        txt = (raw + " " + sub).str.strip()
    else:
        txt = raw

    txt = (
        txt.str.replace("/", " ", regex=False)
           .str.replace("-", " ", regex=False)
           .str.replace("_", " ", regex=False)
    )

    gk_mask = txt.str.contains(r"\bgoalkeeper\b|\bkeeper\b", regex=True, na=False)

    defender_mask = txt.str.contains(
        r"defender|centre back|center back|left back|right back|full back|fullback|sweeper",
        regex=True,
        na=False
    )

    midfielder_mask = txt.str.contains(
        r"midfield|midfielder|defensive midfield|attacking midfield|central midfield|left midfield|right midfield",
        regex=True,
        na=False
    )

    forward_mask = txt.str.contains(
        r"forward|striker|winger|attack|centre forward|center forward|second striker|inside forward",
        regex=True,
        na=False
    )

    broad = np.select(
        [gk_mask, defender_mask, midfielder_mask, forward_mask],
        ["Goalkeeper", "Defender", "Midfielder", "Forward"],
        default="Other"
    )

    return pd.Series(broad, index=raw_position.index, dtype="string")


def clean_profiles(df: pd.DataFrame) -> pd.DataFrame:
    df = standardize_columns(df)

    player_id_col = pick_first_existing(df, ["player_id"])
    player_name_col = pick_first_existing(df, ["player_name", "name"], required=False)
    dob_col = pick_first_existing(df, ["date_of_birth", "dob"], required=False)
    position_col = pick_first_existing(df, ["main_position", "position", "player_main_position"], required=False)
    sub_position_col = pick_first_existing(df, ["player_sub_position", "sub_position"], required=False)
    joined_col = pick_first_existing(df, ["joined"], required=False)
    contract_expires_col = pick_first_existing(df, ["contract_expires"], required=False)
    current_club_col = pick_first_existing(df, ["current_club_id"], required=False)

    out = pd.DataFrame({
        "player_id": df[player_id_col].astype("string").str.strip()
    })

    out["player_name"] = df[player_name_col].astype("string").str.strip() if player_name_col else pd.Series(pd.NA, index=df.index, dtype="string")
    out["date_of_birth"] = parse_date_flex(df[dob_col]) if dob_col else pd.NaT
    out["raw_position"] = df[position_col].astype("string").str.strip() if position_col else pd.Series(pd.NA, index=df.index, dtype="string")
    out["raw_sub_position"] = df[sub_position_col].astype("string").str.strip() if sub_position_col else pd.Series(pd.NA, index=df.index, dtype="string")
    out["joined"] = parse_date_flex(df[joined_col]) if joined_col else pd.NaT
    out["contract_expires"] = parse_date_flex(df[contract_expires_col]) if contract_expires_col else pd.NaT
    out["current_club_id"] = df[current_club_col].astype("string").str.strip() if current_club_col else pd.Series(pd.NA, index=df.index, dtype="string")

    out["broad_position"] = map_broad_position(out["raw_position"], out["raw_sub_position"]).astype("category")

    out = out.dropna(subset=["player_id"]).drop_duplicates(subset=["player_id"]).reset_index(drop=True)
    return out


def clean_market_values(df: pd.DataFrame) -> pd.DataFrame:
    df = standardize_columns(df)

    player_id_col = pick_first_existing(df, ["player_id"])
    value_col = pick_first_existing(df, ["value", "market_value", "market_value_eur"], required=False)
    date_col = pick_first_existing(df, ["date"], required=False)
    date_unix_col = pick_first_existing(df, ["date_unix"], required=False)

    out = pd.DataFrame({
        "player_id": df[player_id_col].astype("string").str.strip()
    })

    if date_col:
        out["market_value_date"] = parse_date_flex(df[date_col])
    elif date_unix_col:
        out["market_value_date"] = parse_unix_flex(df[date_unix_col])
    else:
        raise KeyError("No market value date column found.")

    if value_col:
        out["market_value_eur"] = pd.to_numeric(df[value_col], errors="coerce")
    else:
        raise KeyError("No market value column found.")

    out = out.dropna(subset=["player_id", "market_value_date", "market_value_eur"])
    out = out.drop_duplicates(subset=["player_id", "market_value_date", "market_value_eur"])
    out = out.sort_values(["player_id", "market_value_date"]).reset_index(drop=True)

    return out


def clean_performances(df: pd.DataFrame) -> pd.DataFrame:
    df = standardize_columns(df)

    player_id_col = pick_first_existing(df, ["player_id"])
    season_col = pick_first_existing(df, ["season", "season_name"], required=False)
    competition_id_col = pick_first_existing(df, ["competition_id"], required=False)
    competition_name_col = pick_first_existing(df, ["competition_name"], required=False)
    team_id_col = pick_first_existing(df, ["team_id"], required=False)
    team_name_col = pick_first_existing(df, ["team_name"], required=False)

    out = pd.DataFrame({
        "player_id": df[player_id_col].astype("string").str.strip()
    })

    out["season"] = df[season_col].astype("string").str.strip() if season_col else pd.Series(pd.NA, index=df.index, dtype="string")
    out["season_start_year"] = extract_season_start_year(out["season"]) if season_col else pd.Series(pd.NA, index=df.index, dtype="Int32")
    out["competition_id"] = df[competition_id_col].astype("string").str.strip() if competition_id_col else pd.Series(pd.NA, index=df.index, dtype="string")
    out["competition_name"] = df[competition_name_col].astype("string").str.strip() if competition_name_col else pd.Series(pd.NA, index=df.index, dtype="string")
    out["team_id"] = df[team_id_col].astype("string").str.strip() if team_id_col else pd.Series(pd.NA, index=df.index, dtype="string")
    out["team_name"] = df[team_name_col].astype("string").str.strip() if team_name_col else pd.Series(pd.NA, index=df.index, dtype="string")

    numeric_cols = [
        "nb_in_group", "nb_on_pitch", "goals", "own_goals", "assists", "subed_in", "subed_out",
        "yellow_cards", "second_yellow_cards", "direct_red_cards", "penalty_goals",
        "minutes_played", "goals_conceded", "clean_sheets"
    ]

    for col in numeric_cols:
        if col in df.columns:
            out[col] = to_nullable_int(df[col])
        else:
            out[col] = pd.Series(pd.NA, index=df.index, dtype="Int32")

    out = out.dropna(subset=["player_id"]).drop_duplicates().reset_index(drop=True)
    return out


def build_lifecycle_backbone(
    profiles_clean: pd.DataFrame,
    market_clean: pd.DataFrame,
    min_obs_per_player: int = 3,
    min_age: float = 14.0,
    max_age: float = 45.0
) -> pd.DataFrame:
    keep_cols = ["player_id", "player_name", "date_of_birth", "raw_position", "raw_sub_position", "broad_position"]
    prof = profiles_clean[keep_cols].copy()

    df = market_clean.merge(prof, on="player_id", how="inner")

    df["age_days"] = (df["market_value_date"] - df["date_of_birth"]).dt.days
    df["age_years"] = df["age_days"] / 365.25

    df = df[df["market_value_eur"] > 0].copy()
    df = df[df["age_years"].between(min_age, max_age, inclusive="both")].copy()
    df = df[df["broad_position"].isin(["Goalkeeper", "Defender", "Midfielder", "Forward"])].copy()

    df = df.sort_values(["player_id", "market_value_date"]).reset_index(drop=True)
    df["n_market_value_obs"] = df.groupby("player_id")["market_value_date"].transform("size").astype("Int32")
    df = df[df["n_market_value_obs"] >= min_obs_per_player].copy()

    df["market_value_rank_desc"] = (
        df.groupby("player_id")["market_value_eur"]
          .rank(method="dense", ascending=False)
          .astype("Int32")
    )
    df["is_peak_value_obs"] = (df["market_value_rank_desc"] == 1)

    return df.reset_index(drop=True)


def build_data_dictionary() -> pd.DataFrame:
    rows = [
        ["player_profiles_clean.parquet", "player_id", "Player identifier", "string"],
        ["player_profiles_clean.parquet", "player_name", "Player display name", "string"],
        ["player_profiles_clean.parquet", "date_of_birth", "Date of birth", "datetime64[ns]"],
        ["player_profiles_clean.parquet", "raw_position", "Original position field from source", "string"],
        ["player_profiles_clean.parquet", "raw_sub_position", "Original sub-position field if present", "string"],
        ["player_profiles_clean.parquet", "broad_position", "Mapped broad group: Goalkeeper, Defender, Midfielder, Forward, Other", "category"],

        ["player_market_value_clean.parquet", "player_id", "Player identifier", "string"],
        ["player_market_value_clean.parquet", "market_value_date", "Date of market value observation", "datetime64[ns]"],
        ["player_market_value_clean.parquet", "market_value_eur", "Market value in euros", "float64"],

        ["player_performances_clean.parquet", "player_id", "Player identifier", "string"],
        ["player_performances_clean.parquet", "season", "Season label from source", "string"],
        ["player_performances_clean.parquet", "season_start_year", "Parsed first year of season", "Int32"],
        ["player_performances_clean.parquet", "minutes_played", "Minutes played", "Int32"],
        ["player_performances_clean.parquet", "goals", "Goals scored", "Int32"],
        ["player_performances_clean.parquet", "assists", "Assists", "Int32"],
        ["player_performances_clean.parquet", "clean_sheets", "Clean sheets", "Int32"],

        ["player_lifecycle_backbone.parquet", "player_id", "Player identifier", "string"],
        ["player_lifecycle_backbone.parquet", "market_value_date", "Date of market value observation", "datetime64[ns]"],
        ["player_lifecycle_backbone.parquet", "market_value_eur", "Market value in euros", "float64"],
        ["player_lifecycle_backbone.parquet", "date_of_birth", "Date of birth", "datetime64[ns]"],
        ["player_lifecycle_backbone.parquet", "age_days", "Age in days at market value date", "float64"],
        ["player_lifecycle_backbone.parquet", "age_years", "Age in years at market value date", "float64"],
        ["player_lifecycle_backbone.parquet", "broad_position", "Mapped broad position group", "category"],
        ["player_lifecycle_backbone.parquet", "n_market_value_obs", "Number of market value observations for player", "Int32"],
        ["player_lifecycle_backbone.parquet", "is_peak_value_obs", "Whether this row is tied for player peak market value", "bool"],
        ["player_lifecycle_backbone.parquet", "market_value_rank_desc", "Dense descending rank of market value within each player career history", "Int32"],
    ]

    return pd.DataFrame(rows, columns=["dataset", "column_name", "description", "dtype"])

Overwriting /content/drive/MyDrive/football_lifecycle/src/a_preprocessing_utils.py


## 3. Load Raw Data

In [4]:
import sys
import os
import io
import time
import cProfile
import pstats
import pandas as pd
import numpy as np

sys.path.append(os.path.abspath('..'))

from src.a_preprocessing_utils import (
    clean_profiles,
    clean_market_values,
    clean_performances,
    build_lifecycle_backbone,
    build_data_dictionary,
)

In [5]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/football_lifecycle")

raw_profiles_path = PROJECT_ROOT / "data" / "raw" / "player_profiles.csv"
raw_market_path = PROJECT_ROOT / "data" / "raw" / "player_market_value.csv"
raw_performance_path = PROJECT_ROOT / "data" / "raw" / "player_performances.txt"

profiles_raw = pd.read_csv(raw_profiles_path, low_memory=False)
market_raw = pd.read_csv(raw_market_path, low_memory=False)
performances_raw = pd.read_csv(raw_performance_path, low_memory=False)

print("profiles_raw shape:", profiles_raw.shape)
print("market_raw shape:", market_raw.shape)
print("performances_raw shape:", performances_raw.shape)

profiles_raw shape: (92671, 34)
market_raw shape: (901429, 3)
performances_raw shape: (1878719, 20)


## 4. Raw Data Audit
This section checks the structure, completeness, and schema consistency of the three core raw tables.

In [6]:
print("profiles columns:")
print(profiles_raw.columns.tolist())

print("\nmarket columns:")
print(market_raw.columns.tolist())

print("\nperformances columns:")
print(performances_raw.columns.tolist())

profiles columns:
['player_id', 'player_slug', 'player_name', 'player_image_url', 'name_in_home_country', 'date_of_birth', 'place_of_birth', 'country_of_birth', 'height', 'citizenship', 'is_eu', 'position', 'main_position', 'foot', 'current_club_id', 'current_club_name', 'joined', 'contract_expires', 'outfitter', 'social_media_url', 'player_agent_id', 'player_agent_name', 'contract_option', 'date_of_last_contract_extension', 'on_loan_from_club_id', 'on_loan_from_club_name', 'contract_there_expires', 'second_club_url', 'second_club_name', 'third_club_url', 'third_club_name', 'fourth_club_url', 'fourth_club_name', 'date_of_death']

market columns:
['player_id', 'date_unix', 'value']

performances columns:
['player_id', 'season_name', 'competition_id', 'competition_name', 'team_id', 'team_name', 'nb_in_group', 'nb_on_pitch', 'goals', 'assists', 'own_goals', 'subed_in', 'subed_out', 'yellow_cards', 'second_yellow_cards', 'direct_red_cards', 'penalty_goals', 'minutes_played', 'goals_concede

In [7]:
def quick_audit(df, name):
    print(f"\n===== {name} =====")
    print("shape:", df.shape)
    print("duplicate rows:", df.duplicated().sum())
    print("missing by column, top 20:")
    print(df.isna().sum().sort_values(ascending=False).head(20))
    print("dtypes, top 20:")
    print(df.dtypes.head(20))

quick_audit(profiles_raw, "profiles_raw")
quick_audit(market_raw, "market_raw")
quick_audit(performances_raw, "performances_raw")


===== profiles_raw =====
shape: (92671, 34)
duplicate rows: 0
missing by column, top 20:
fourth_club_url                    92670
fourth_club_name                   92670
third_club_url                     92629
third_club_name                    92629
date_of_death                      92399
second_club_url                    91693
second_club_name                   91693
contract_there_expires             90165
contract_option                    89100
on_loan_from_club_id               88913
on_loan_from_club_name             88913
outfitter                          86713
date_of_last_contract_extension    84308
social_media_url                   72334
player_agent_id                    54614
contract_expires                   54005
player_agent_name                  49791
name_in_home_country               43018
foot                               23488
country_of_birth                   16645
dtype: int64
dtypes, top 20:
player_id                 int64
player_slug              obje

## 5. Clean Core Tables

In [8]:
profiles_clean = clean_profiles(profiles_raw)
market_clean = clean_market_values(market_raw)
performances_clean = clean_performances(performances_raw)

print("profiles_clean shape:", profiles_clean.shape)
print("market_clean shape:", market_clean.shape)
print("performances_clean shape:", performances_clean.shape)

profiles_clean shape: (92671, 9)
market_clean shape: (901429, 3)
performances_clean shape: (1878719, 21)


## 6. Validate Cleaned Tables

In [9]:
display(profiles_clean.head())
display(market_clean.head())
display(performances_clean.head())

,player_id,player_name,date_of_birth,raw_position,raw_sub_position,joined,contract_expires,current_club_id,broad_position
0,1,Silvio Adzic (1),1980-09-23,Attack,<NA>,2017-07-01,NaT,123,Forward
1,100011,Éverton Silva (100011),1988-08-04,Defender,<NA>,2025-03-01,NaT,515,Defender
2,10,Miroslav Klose (10),1978-06-09,Attack,<NA>,2016-07-01,NaT,123,Forward
3,10001,John Thompson (10001),1981-10-12,Defender,<NA>,2013-07-01,NaT,123,Defender
4,100001,Carlos Auzqui (100001),1991-03-16,Attack,<NA>,2025-01-30,2025-12-31,14554,Forward


,player_id,market_value_date,market_value_eur
0,1,2004-10-03,250000.0
1,1,2007-06-18,200000.0
2,1,2009-04-22,0.0
3,10,2004-10-03,7000000.0
4,10,2005-01-06,9000000.0


,player_id,season,season_start_year,competition_id,competition_name,team_id,team_name,nb_in_group,nb_on_pitch,goals,...,assists,subed_in,subed_out,yellow_cards,second_yellow_cards,direct_red_cards,penalty_goals,minutes_played,goals_conceded,clean_sheets
0,1,08/09,2008,OBLG,NOFV-Oberliga Süd,4825,FC Eilenburg,9,9,0,...,0,0,2,0,0,1,0,<NA>,0,0
1,1,07/08,2007,RS,Regionalliga Süd,1526,FSV Ludwigshafen Oggersheim,22,22,1,...,0,3,8,1,0,0,0,1580,0,0
2,1,06/07,2006,L2,2. Bundesliga,996,TuS Koblenz,10,4,0,...,0,4,0,0,0,0,0,<NA>,0,0
3,1,06/07,2006,DFB,DFB-Pokal,996,TuS Koblenz,1,0,0,...,0,0,0,0,0,0,0,<NA>,0,0
4,1,05/06,2005,L2,2. Bundesliga,66,SpVgg Unterhaching,26,14,1,...,1,12,1,1,0,0,0,388,0,0


In [10]:
print("Broad position counts:")
print(profiles_clean["broad_position"].value_counts(dropna=False))

print("\nProfiles missing DOB:")
print(profiles_clean["date_of_birth"].isna().sum())

print("\nMarket missing date:")
print(market_clean["market_value_date"].isna().sum())

print("\nMarket missing value:")
print(market_clean["market_value_eur"].isna().sum())

print("\nPerformance missing season:")
print(performances_clean["season"].isna().sum())

print("\nPerformance numeric summary:")
display(
    performances_clean[["minutes_played", "goals", "assists", "clean_sheets"]]
    .describe(include="all")
)

Broad position counts:
broad_position
Defender      29733
Midfielder    28373
Forward       24287
Goalkeeper    10276
Other             2
Name: count, dtype: int64

Profiles missing DOB:
1006

Market missing date:
0

Market missing value:
0

Performance missing season:
0

Performance numeric summary:


,minutes_played,goals,assists,clean_sheets
count,708161.0,1740517.0,1878719.0,1878719.0
mean,606.18123,0.952719,0.446945,0.191107
std,681.591337,2.283932,1.24737,1.204661
min,1.0,0.0,0.0,0.0
25%,178.0,0.0,0.0,0.0
50%,337.0,0.0,0.0,0.0
75%,758.0,1.0,0.0,0.0
max,4590.0,50.0,29.0,29.0


## 7. Save Clean Intermediate Files

In [11]:
profiles_clean_path = PROJECT_ROOT / "data" / "interim" / "player_profiles_clean.parquet"
market_clean_path = PROJECT_ROOT / "data" / "interim" / "player_market_value_clean.parquet"
performances_clean_path = PROJECT_ROOT / "data" / "interim" / "player_performances_clean.parquet"

profiles_clean.to_parquet(profiles_clean_path, index=False)
market_clean.to_parquet(market_clean_path, index=False)
performances_clean.to_parquet(performances_clean_path, index=False)

print("Saved:")
print(profiles_clean_path)
print(market_clean_path)
print(performances_clean_path)

Saved:
/content/drive/MyDrive/football_lifecycle/data/interim/player_profiles_clean.parquet
/content/drive/MyDrive/football_lifecycle/data/interim/player_market_value_clean.parquet
/content/drive/MyDrive/football_lifecycle/data/interim/player_performances_clean.parquet


## 8. Build Frozen Backbone Dataset
This section constructs the shared market value backbone that will be used by later research-question notebooks.

In [12]:
backbone = build_lifecycle_backbone(
    profiles_clean=profiles_clean,
    market_clean=market_clean,
    min_obs_per_player=3,
    min_age=14.0,
    max_age=45.0
)

print("backbone shape:", backbone.shape)
display(backbone.head())

backbone shape: (836635, 13)


,player_id,market_value_date,market_value_eur,player_name,date_of_birth,raw_position,raw_sub_position,broad_position,age_days,age_years,n_market_value_obs,market_value_rank_desc,is_peak_value_obs
0,10,2004-10-03,7000000.0,Miroslav Klose (10),1978-06-09,Attack,<NA>,Forward,9613.0,26.318960,23,9,False
1,10,2005-01-06,9000000.0,Miroslav Klose (10),1978-06-09,Attack,<NA>,Forward,9708.0,26.579055,23,7,False
2,10,2005-05-04,12000000.0,Miroslav Klose (10),1978-06-09,Attack,<NA>,Forward,9826.0,26.902122,23,6,False
3,10,2005-09-29,15000000.0,Miroslav Klose (10),1978-06-09,Attack,<NA>,Forward,9974.0,27.307324,23,5,False
4,10,2006-01-08,20000000.0,Miroslav Klose (10),1978-06-09,Attack,<NA>,Forward,10075.0,27.583847,23,3,False


In [13]:
print("Backbone broad position counts:")
print(backbone["broad_position"].value_counts())

print("\nAge summary:")
print(backbone["age_years"].describe())

print("\nPlayers in backbone:")
print(backbone["player_id"].nunique())

print("\nRows per player summary:")
print(backbone["n_market_value_obs"].describe())

print("\nAny negative ages?")
print((backbone["age_years"] < 0).sum())

print("\nAny duplicate player-date rows?")
print(backbone.duplicated(subset=["player_id", "market_value_date"]).sum())

Backbone broad position counts:
broad_position
Defender      275452
Midfielder    246939
Forward       227838
Goalkeeper     86406
Other              0
Name: count, dtype: int64

Age summary:
count    836635.000000
mean         25.704193
std           4.739146
min          15.597536
25%          21.900068
50%          25.169062
75%          29.048597
max          44.925394
Name: age_years, dtype: float64

Players in backbone:
58944

Rows per player summary:
count     836635.0
mean     20.185007
std       9.761228
min            3.0
25%           12.0
50%           20.0
75%           27.0
max           55.0
Name: n_market_value_obs, dtype: Float64

Any negative ages?
0

Any duplicate player-date rows?
0


## 9. Backbone Sanity Checks

In [14]:
backbone["n_market_value_obs"] = backbone["n_market_value_obs"].astype("Int32")

In [15]:
n_profiles_raw = profiles_raw["player_id"].nunique()
n_profiles_clean = profiles_clean["player_id"].nunique()
n_market_players = market_clean["player_id"].nunique()
n_backbone_players = backbone["player_id"].nunique()

print("Unique players in raw profiles:", n_profiles_raw)
print("Unique players in cleaned profiles:", n_profiles_clean)
print("Unique players in cleaned market table:", n_market_players)
print("Unique players in backbone:", n_backbone_players)

Unique players in raw profiles: 92671
Unique players in cleaned profiles: 92671
Unique players in cleaned market table: 69441
Unique players in backbone: 58944


## 10. Sample Attrition Summary

In [16]:
attrition_summary = pd.DataFrame({
    "stage": [
        "raw_profiles",
        "cleaned_profiles",
        "cleaned_market_unique_players",
        "backbone_final_players"
    ],
    "n_unique_players": [
        n_profiles_raw,
        n_profiles_clean,
        n_market_players,
        n_backbone_players
    ]
})

display(attrition_summary)

attrition_summary_path = PROJECT_ROOT / "outputs" / "tables" / "A_attrition_summary.csv"
attrition_summary.to_csv(attrition_summary_path, index=False)
print(attrition_summary_path)

,stage,n_unique_players
0,raw_profiles,92671
1,cleaned_profiles,92671
2,cleaned_market_unique_players,69441
3,backbone_final_players,58944


/content/drive/MyDrive/football_lifecycle/outputs/tables/A_attrition_summary.csv


## 11. Save Final Backbone and Data Dictionary

In [17]:
backbone_path = PROJECT_ROOT / "data" / "processed" / "player_lifecycle_backbone.parquet"
backbone.to_parquet(backbone_path, index=False)

print("Saved backbone to:")
print(backbone_path)

Saved backbone to:
/content/drive/MyDrive/football_lifecycle/data/processed/player_lifecycle_backbone.parquet


In [18]:
data_dictionary = build_data_dictionary()
data_dictionary_path = PROJECT_ROOT / "outputs" / "tables" / "data_dictionary.csv"

data_dictionary.to_csv(data_dictionary_path, index=False)

print("Saved data dictionary to:")
print(data_dictionary_path)

display(data_dictionary)

Saved data dictionary to:
/content/drive/MyDrive/football_lifecycle/outputs/tables/data_dictionary.csv


,dataset,column_name,description,dtype
0,player_profiles_clean.parquet,player_id,Player identifier,string
1,player_profiles_clean.parquet,player_name,Player display name,string
2,player_profiles_clean.parquet,date_of_birth,Date of birth,datetime64[ns]
3,player_profiles_clean.parquet,raw_position,Original position field from source,string
4,player_profiles_clean.parquet,raw_sub_position,Original sub-position field if present,string
5,player_profiles_clean.parquet,broad_position,"Mapped broad group: Goalkeeper, Defender, Midf...",category
6,player_market_value_clean.parquet,player_id,Player identifier,string
7,player_market_value_clean.parquet,market_value_date,Date of market value observation,datetime64[ns]
8,player_market_value_clean.parquet,market_value_eur,Market value in euros,float64
9,player_performances_clean.parquet,player_id,Player identifier,string


## 12. Profiling Pipeline
This section profiles the preprocessing pipeline to identify early bottlenecks and document low-risk optimizations.

In [19]:
def run_a_pipeline(raw_profiles_path, raw_market_path, raw_performance_path):
    profiles_raw_local = pd.read_csv(raw_profiles_path, low_memory=False)
    market_raw_local = pd.read_csv(raw_market_path, low_memory=False)
    performances_raw_local = pd.read_csv(raw_performance_path, low_memory=False)

    profiles_clean_local = clean_profiles(profiles_raw_local)
    market_clean_local = clean_market_values(market_raw_local)
    performances_clean_local = clean_performances(performances_raw_local)

    backbone_local = build_lifecycle_backbone(
        profiles_clean=profiles_clean_local,
        market_clean=market_clean_local,
        min_obs_per_player=3,
        min_age=14.0,
        max_age=45.0
    )

    return profiles_clean_local, market_clean_local, performances_clean_local, backbone_local

In [20]:
profiler = cProfile.Profile()

start = time.perf_counter()
profiler.enable()

_ = run_a_pipeline(
    raw_profiles_path,
    raw_market_path,
    raw_performance_path
)

profiler.disable()
elapsed = time.perf_counter() - start

s = io.StringIO()
stats = pstats.Stats(profiler, stream=s).sort_stats("cumtime")
stats.print_stats(25)

profile_output = s.getvalue()

print(f"End-to-end pipeline runtime: {elapsed:.2f} seconds")
print(profile_output[:4000])

End-to-end pipeline runtime: 38.81 seconds
         39509934 function calls (39508531 primitive calls) in 38.723 seconds

   Ordered by: cumulative time
   List reduced from 1251 to 25 due to restriction <25>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.199    0.199   24.685   24.685 /content/drive/MyDrive/football_lifecycle/src/a_preprocessing_utils.py:168(clean_performances)
       41    0.000    0.000   16.938    0.413 /usr/local/lib/python3.12/dist-packages/pandas/core/strings/accessor.py:129(wrapper)
       18    0.000    0.000   12.204    0.678 /usr/local/lib/python3.12/dist-packages/pandas/core/strings/accessor.py:2115(strip)
       18    0.000    0.000   12.198    0.678 /usr/local/lib/python3.12/dist-packages/pandas/core/strings/object_array.py:450(_str_strip)
       11    3.019    0.274   10.880    0.989 /usr/local/lib/python3.12/dist-packages/pandas/core/arrays/string_.py:604(_str_map)
       36    8.690    0.241    8.949    0.249 {b

## 13. Export Deliverables

In [21]:
clean_perf_seconds = 23.81
read_csv_seconds = 7.08
backbone_seconds = 4.14

profile_md = f"""
# A Preprocessing Profile

## Scope
This profiling pass covers:
- raw data loading
- table cleaning
- backbone merge
- vectorized age construction

## End-to-end runtime
{elapsed:.2f} seconds

## Main bottleneck
The dominant preprocessing bottleneck in the A-stage pipeline is the performance-table cleaning step, which is expected because the performance table is the largest of the three core inputs.

In the first cProfile pass, the full A-stage pipeline ran in about {elapsed:.1f} seconds. The largest cumulative-cost function was clean_performances() at about {clean_perf_seconds:.1f} seconds, followed by CSV loading at about {read_csv_seconds:.1f} seconds and lifecycle backbone construction at about {backbone_seconds:.1f} seconds.

## Early optimizations already applied
- vectorized age calculation
- reduced column set after cleaning
- categorical encoding for broad_position
- nullable integer types for performance metrics
- avoidance of row-wise Python loops in backbone construction

## Interpretation
These results suggest that the first-stage optimization priority should remain focused on large-table preprocessing and column-wise cleaning efficiency rather than on the backbone merge itself. The backbone construction step is nontrivial but not the dominant cost at this stage.

## Note
This profiling note should be handed to D for integration into the final technical execution summary.
"""

profile_note_path = PROJECT_ROOT / "outputs" / "profiling" / "A_preprocessing_profile.md"

with open(profile_note_path, "w", encoding="utf-8") as f:
    f.write(profile_md)

print("Saved profiling note to:")
print(profile_note_path)

Saved profiling note to:
/content/drive/MyDrive/football_lifecycle/outputs/profiling/A_preprocessing_profile.md


In [22]:
methods_md = """
# A Data and Preprocessing Methods

## Data source
We used the public Transfermarkt-based football dataset from the salimt GitHub repository.
The A-stage pipeline focused on three core files:
- player_profiles.csv
- player_market_value.csv
- player_performances.csv

## Preprocessing overview
We standardized column names, parsed date fields, validated key identifiers, reduced each table to analysis-relevant columns, and saved cleaned intermediate parquet files.

## Position handling
Player positions were mapped into four broad groups:
Goalkeeper, Defender, Midfielder, and Forward.
Records with missing or unmappable position information were excluded from the frozen backbone.

## Merge strategy
The frozen lifecycle backbone was built by merging cleaned player profiles with cleaned market value observations on player_id.
The performance table was intentionally kept separate at this stage, because direct merging on player_id alone would create many-to-many duplication without time alignment.

## Age construction
Age was computed vectorially as the difference between market value date and date of birth, divided by 365.25.

## Backbone filters
The frozen backbone retained observations with:
- positive market value
- age between 14 and 45
- broad position in the four main groups
- at least 3 market value observations per player

## Data quality notes
Date of birth was missing for a small subset of player profiles.
The performance table had substantial missingness in minutes_played, so missing values were preserved rather than aggressively imputed.
"""

with open(PROJECT_ROOT / "report" / "A_data_preprocessing_methods.md", "w", encoding="utf-8") as f:
    f.write(methods_md)

In [23]:
final_files = [
    PROJECT_ROOT / "data" / "interim" / "player_profiles_clean.parquet",
    PROJECT_ROOT / "data" / "interim" / "player_market_value_clean.parquet",
    PROJECT_ROOT / "data" / "interim" / "player_performances_clean.parquet",
    PROJECT_ROOT / "data" / "processed" / "player_lifecycle_backbone.parquet",
    PROJECT_ROOT / "outputs" / "tables" / "data_dictionary.csv",
    PROJECT_ROOT / "outputs" / "profiling" / "A_preprocessing_profile.md",
    PROJECT_ROOT / "report" / "A_data_preprocessing_methods.md",
]

for fp in final_files:
    print(fp.name, "exists:", fp.exists())

player_profiles_clean.parquet exists: True
player_market_value_clean.parquet exists: True
player_performances_clean.parquet exists: True
player_lifecycle_backbone.parquet exists: True
data_dictionary.csv exists: True
A_preprocessing_profile.md exists: True
A_data_preprocessing_methods.md exists: True
